# Pareja A · Idea 7a — Nuevos codificadores para el ensamble

Vectores con CLIP L/14 (`openai/clip-vit-large-patch14`, 768) y SigLIP base (`google/siglip-base-patch16-224`, 768). Opcionales: SigLIP so400m (1152) y DINOv2 (768, sin texto). SigLIP necesita `AutoModel`/`AutoProcessor` (no `CLIPModel`, no tiene `projection_dim`). Primero los de recorte central; las variantes, solo del mejor. **Listo cuando** los vectores están en `poster2movie_clasificador/vectores/` para B y C.

**Dónde se programa:** `codificadores.py`: pasar `clip_l14` y `siglip_base` de `CODIFICADORES_PENDIENTES` a `CODIFICADORES`, y una clase para SigLIP registrada en `crear_codificador`. La lógica va en los `.py` (se revisa en el Pull Request); este
notebook solo la corre. **Listo cuando** su prueba pasa (quiten el `xfail` de `test_idea7a` en
`tests/test_pendientes.py`) y el resultado en validación está en el registro.

Antes, una vez para todo el equipo: `00_reproducir_base` (la base `v0.1.0`).

## Preparar el entorno

**GPU (T4)**: *Entorno de ejecución → Cambiar tipo de entorno de ejecución*. La cuota gratis se acaba: calculen primero lo prioritario y repartan el cómputo entre sus cuentas.

- `P2M_DATOS`: dataset compartido (solo lectura). `P2M_SALIDAS`: carpeta **compartida del equipo**
  (`poster2movie_clasificador`), donde quedan vectores, modelos y predicciones para las otras parejas.
  Agreguen las dos a *Mi unidad* como acceso directo.
- Trabaja en la rama **`A-codificadores`** (se crea desde `main` si no existe).

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
os.environ["P2M_DATOS"] = "/content/drive/MyDrive/poster2movie_datos"            # solo lectura
os.environ["P2M_SALIDAS"] = "/content/drive/MyDrive/poster2movie_clasificador"   # compartida del equipo
REPO = "11909/Poster2Movie-Clasificador"
RAMA = "A-codificadores"
DESTINO = "/content/clasificador"
if not os.path.isdir(DESTINO):
    !git clone -q --recurse-submodules https://github.com/{REPO}.git {DESTINO}
%cd {DESTINO}
!git fetch -q origin
!git checkout -q {RAMA} 2>/dev/null || git checkout -q -b {RAMA} origin/main
!git pull -q origin {RAMA} 2>/dev/null; git submodule update -q --init
!pip install -q -r requirements.txt
!nvidia-smi -L

In [ ]:
# ¿ya pasa la prueba de la idea? (mientras esté pendiente sale como "xfailed")
!python -m pytest -q tests/test_pendientes.py -k idea7a -rxX

## 1. Vectores (GPU; L/14 tarda bastante más que B/32)

In [ ]:
!python -m clasificador.codificadores clip_l14
!python -m clasificador.codificadores siglip_base

## 2. Una cabeza por codificador (también serán miembros del ensamble de la Pareja C)

In [ ]:
# entrenar en train, evaluar en VALIDACIÓN y comparar contra base (los 4 débiles salen primero)
for nombre in ["A7_l14", "A7_siglip"]:
    !python -m clasificador.entrenar experimentos/{nombre}.yaml
    !python -m clasificador.evaluar {nombre}
    !python -m clasificador.comparar base {nombre}

## Subir a la rama `A-codificadores`

En Colab el repo se borra al cerrar: suban `resultados/` antes. Se sube a **`A-codificadores`**, nunca a
`main`; luego abran el Pull Request (lo revisa alguien de **otra** pareja). Necesitan un token de
GitHub con permiso de escritura (*Settings → Developer settings → Personal access tokens → Tokens (classic)*, con el permiso `repo` (los *fine-grained* no sirven: el repo es de otra cuenta personal)).

In [ ]:
from getpass import getpass
!git config user.name "{input('Tu nombre: ')}"
!git config user.email "{input('Tu correo de GitHub: ')}"
!git add experimentos resultados clasificador tests
!git commit -q -m "{RAMA}: {input('Qué hiciste (mensaje del commit): ')}"
token = getpass("Token de GitHub: ")
!git push -q https://{token}@github.com/{REPO}.git HEAD:{RAMA}
del token
print(f"Abre el Pull Request: https://github.com/{REPO}/compare/main...{RAMA}")